# Rossmann Retail - Data Overview and SQL Integration

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("Python-Pfad:", sys.executable)
print("Python-Version:", sys.version.split()[0])
print("Pandas-Version:", pd.__version__)
print("Scikit-learn-Version:", sklearn.__version__)

Python-Pfad: c:\Users\moonp\Documents\Beruflich\Machine Learning Projekte\Rossmann_retail\.venv\Scripts\python.exe
Python-Version: 3.14.7
Pandas-Version: 3.1.0
Scikit-learn-Version: 1.9.1


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Projektordner bestimmen:
# Das Notebook startet normalerweise im Projektordner oder in "notebooks".
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Datenordner:", DATA_DIR.resolve())

# CSV-Dateien einlesen
train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=["Date"],
    dtype={"StateHoliday": "string"},
)

stores = pd.read_csv(DATA_DIR / "store.csv")

Datenordner: C:\Users\moonp\Documents\Beruflich\Machine Learning Projekte\Rossmann_retail\data\raw


In [3]:
print("Umsatzdaten – Zeilen und Spalten:", train.shape)
print("Filialdaten – Zeilen und Spalten:", stores.shape)

print(
    "Zeitraum:",
    train["Date"].min().date(),
    "bis",
    train["Date"].max().date(),
)

print("Anzahl Filialen in den Umsatzdaten:", train["Store"].nunique())

display(train.head())
display(stores.head())

Umsatzdaten – Zeilen und Spalten: (1017209, 9)
Filialdaten – Zeilen und Spalten: (1115, 10)
Zeitraum: 2013-01-01 bis 2015-07-31
Anzahl Filialen in den Umsatzdaten: 1115


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [ ]:
def overview(df):
    '''
    Erstelle einen Überblick über einige wichtige Eigenschaften der Spalten eines DataFrames.
    VARs
        df: Der zu betrachtende DataFrame
    RETURNS:
        None
    '''
    df = df.copy()
    display(pd.DataFrame({'dtype': df.dtypes,                                   # Dtypes
                          'total': df.count(),                                  # Anzahl valider Einträge
                          'missing_n': df.isna().sum(),                         # Anzahl nan-Einträge
                          'missing_%': df.isna().mean()*100,                    # Anteil fehlender Daten in %
                          'uniques_n': df.nunique(),                            # Kardinalität (Anzahl einzigartiger Werte)
                          'uniques': [df[col].unique() for col in df.columns]   # Auflistung einzigartiger Werte
                          
                         }))
    
    

In [10]:
overview(stores)
overview(train)

print(
    "Duplicate store-date combinations:",
    train.duplicated(subset=["Store", "Date"]).sum(),
)

print(
    "Duplicate store IDs in store.csv:",
    stores.duplicated(subset=["Store"]).sum(),
)

,dtype,total,missing_n,missing_%,uniques_n,uniques
Store,int64,1115,0,0.000000,1115,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
StoreType,str,1115,0,0.000000,4,"[c, a, d, b]"
Assortment,str,1115,0,0.000000,3,"[a, c, b]"
CompetitionDistance,float64,1112,3,0.269058,654,"[1270.0, 570.0, 14130.0, 620.0, 29910.0, 310.0..."
CompetitionOpenSinceMonth,float64,761,354,31.748879,12,"[9.0, 11.0, 12.0, 4.0, 10.0, 8.0, nan, 3.0, 6...."
CompetitionOpenSinceYear,float64,761,354,31.748879,23,"[2008.0, 2007.0, 2006.0, 2009.0, 2015.0, 2013...."
Promo2,int64,1115,0,0.000000,2,"[0, 1]"
Promo2SinceWeek,float64,571,544,48.789238,24,"[nan, 13.0, 14.0, 1.0, 45.0, 40.0, 26.0, 22.0,..."
Promo2SinceYear,float64,571,544,48.789238,7,"[nan, 2010.0, 2011.0, 2012.0, 2009.0, 2014.0, ..."
PromoInterval,str,571,544,48.789238,3,"[nan, Jan,Apr,Jul,Oct, Feb,May,Aug,Nov, Mar,Ju..."


,dtype,total,missing_n,missing_%,uniques_n,uniques
Store,int64,1017209,0,0.0,1115,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
DayOfWeek,int64,1017209,0,0.0,7,"[5, 4, 3, 2, 1, 7, 6]"
Date,datetime64[us],1017209,0,0.0,942,"[2015-07-31 00:00:00, 2015-07-30 00:00:00, 201..."
Sales,int64,1017209,0,0.0,21734,"[5263, 6064, 8314, 13995, 4822, 5651, 15344, 8..."
Customers,int64,1017209,0,0.0,4086,"[555, 625, 821, 1498, 559, 589, 1414, 833, 687..."
Open,int64,1017209,0,0.0,2,"[1, 0]"
Promo,int64,1017209,0,0.0,2,"[1, 0]"
StateHoliday,string,1017209,0,0.0,4,"[0, a, b, c]"
SchoolHoliday,int64,1017209,0,0.0,2,"[1, 0]"


Duplicate store-date combinations: 0
Duplicate store IDs in store.csv: 0


In [8]:
promo_check = stores.groupby("Promo2").agg(
    Filialen=("Store", "size"),
    Startwoche_fehlt=("Promo2SinceWeek", lambda s: s.isna().sum()),
    Startjahr_fehlt=("Promo2SinceYear", lambda s: s.isna().sum()),
    Intervall_fehlt=("PromoInterval", lambda s: s.isna().sum()),
)

display(promo_check)

,Filialen,Startwoche_fehlt,Startjahr_fehlt,Intervall_fehlt
Promo2,,,,
0,544,544,544,544
1,571,0,0,0


missing data in "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval" only for stores which did not take part in promo2. So there is no information regarding start date or promo interval, because it is not applicable

In [9]:
import sqlite3
from contextlib import closing

DB_PATH = PROJECT_ROOT / "data" / "rossmann.sqlite"

with closing(sqlite3.connect(DB_PATH)) as conn:
    train.to_sql(
        "sales",
        conn,
        if_exists="replace",
        index=False,
        chunksize=10000,
    )

    stores.to_sql(
        "stores",
        conn,
        if_exists="replace",
        index=False,
    )

    # Eindeutigkeit der Schlüssel sicherstellen.
    conn.execute("""
        CREATE UNIQUE INDEX ux_sales_store_date
        ON sales (Store, Date)
    """)

    conn.execute("""
        CREATE UNIQUE INDEX ux_stores_store
        ON stores (Store)
    """)

    conn.commit()

print("Datenbank angelegt:", DB_PATH)

Datenbank angelegt: c:\Users\moonp\Documents\Beruflich\Machine Learning Projekte\Rossmann_retail\data\rossmann.sqlite
